<a href="https://colab.research.google.com/github/cbonnin88/MapleFit/blob/main/MapleFit_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.1 MB/s eta 0:00:00


In [15]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

In [26]:
%%writefile app.py
import streamlit as st
from google.cloud import bigquery
import pandas as pd


# 1. Setup BigQuery Client
project_id = 'product-analytics-494706'
client = bigquery.Client(project=project_id)

st.set_page_config(page_title = 'Maplefit Intelligence: Feature Recommendations',layout='wide')

st.title('🍁 MapleFit Product Intelligence')
st.markdown('### User Feature Recommendation Engine')

# 2. Get the Data from my dbt Mart
@st.cache_data
def load_data():
  query = "SELECT * FROM `product-analytics-494706.dbt_maplefit.fct_active_users`"
  return client.query(query).to_dataframe()

df_maple = load_data()

# 3. Sidebar Filter
st.sidebar.header('Filter Segment')
country = st.sidebar.selectbox('Select Country',df_maple['country'].unique())
tier = st.sidebar.selectbox('Select Tier',df_maple['membership_tier'].unique())

filtered_df = df_maple[(df_maple['country'] == country) & (df_maple['membership_tier'] == tier)]

# 4. The Recommendation Logic
st.write(f'Showing results for **{tier}** users in **{country}**')

# Selecting a user to 'coach'
user_id = st.selectbox('Select a User ID to analyze', filtered_df['user_id'].unique())
user_row = df_maple[df_maple['user_id'] == user_id].iloc[0]

# Logic based on the users data
st.subheader(f'Strategy for User: {user_id}')
col1,col2 = st.columns(2)

with col1:
  st.metric('Total Events',int(user_row['total_events']))
  st.metric('Revenue (CAD)',f'${user_row['revenue_cad']:.2f}')

with col2:
  # Custom Recommendation Logic
  if user_row['total_events'] < 10:
    st.error('⚠️ HIGH CHURN RISK')
    st.write('**Recommendation:** Trigger New Workout push notification. This user hasnt hit the 10-event stickiness threshold')
  elif user_row['membership_tier'] == 'Free' and user_row['total_events'] > 30:
    st.success('💎 UPSELL CANDIDATE')
    st.write('**Recommendation:** offer a 15% discount on Premium. They are a power user on the free tier')
  else:
    st.info('✅ ENGAGED USER')
    st.write('**Recommendation:** Suggest Meal Logging feature to diversify their usrage profile')

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os
from google.colab import userdata

ngrok.kill()

ngrok_auth_token = userdata.get('ngrok_token')
ngrok.set_auth_token(ngrok_auth_token)

public_url = ngrok.connect(8501,proto='http')
print('Maplefit Intelligence App is live at: ', {public_url})

!streamlit run app.py --server.port 8501 &

Maplefit Intelligence App is live at:  {<NgrokTunnel: "https://ab04-34-126-105-171.ngrok-free.app" -> "http://localhost:8501">}


2026-05-13 07:19:46.209 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.105.171:8501

